In [ ]:
import numpy as np
import pandas as pd
import pickle as pkl

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score

import mlflow
import mlflow.sklearn

In [ ]:

# tone centroid
tone_centroids = pkl.load(open("../../data_csv/tone_vectors.pkl", "rb"))

# tone metadata
tone_ids = pd.read_csv("../../data_csv/tone_metadata.csv")["tone_id"].tolist()

# rule-based 파라미터 테이블
tone_param_df = pd.read_csv("../../data_csv/tone_centroid_profile.csv")

In [ ]:
rows = []

for tone_id in tone_ids:
    vec = tone_centroids[tone_id]   # 🔥 핵심 수정
    params = tone_param_df[tone_param_df["tone_id"] == tone_id].iloc[0]

    row = {
        "tone_id": tone_id,
        **{f"v_{j}": vec[j] for j in range(len(vec))},
        "proof_level": params["proof_level"],
        "emotion_level": params["emotion_level"],
        "cta_strength": params["cta_strength"],
        "sentence_len": params["sentence_len"]
    }
    rows.append(row)

df = pd.DataFrame(rows)
df.head()

,tone_id,v_0,v_1,v_2,v_3,v_4,v_5,v_6,v_7,v_8,...,v_762,v_763,v_764,v_765,v_766,v_767,proof_level,emotion_level,cta_strength,sentence_len
0,Scientific,-0.010791,0.037765,0.007156,0.018408,0.052377,0.033443,0.05654,-0.021086,0.03585,...,-0.015562,0.027763,-0.07111,0.021353,0.031005,-0.022343,high,low,mid,short
1,Emotional,-0.010791,0.037765,0.007156,0.018408,0.052377,0.033443,0.05654,-0.021086,0.03585,...,-0.015562,0.027763,-0.07111,0.021353,0.031005,-0.022343,low,high,low,mid
2,Luxury,-0.010791,0.037765,0.007156,0.018408,0.052377,0.033443,0.05654,-0.021086,0.03585,...,-0.015562,0.027763,-0.07111,0.021353,0.031005,-0.022343,mid,mid,mid,short
3,Casual,-0.010791,0.037765,0.007156,0.018408,0.052377,0.033443,0.05654,-0.021086,0.03585,...,-0.015562,0.027763,-0.07111,0.021353,0.031005,-0.022343,low,mid,high,very_short


In [ ]:
X = df.filter(regex="^v_").values

y_proof   = df["proof_level"].values
y_emotion = df["emotion_level"].values
y_cta     = df["cta_strength"].values

In [ ]:
mlflow.set_experiment("agent10_tone_param_adjust")

with mlflow.start_run(run_name="ridge_regression"):

    model = Ridge(alpha=1.0)
    model.fit(X, y_proof)

    preds = model.predict(X)
    rmse = mean_squared_error(y_proof, preds, squared=False)

    mlflow.log_param("model", "Ridge")
    mlflow.log_metric("rmse_proof", rmse)
    mlflow.sklearn.log_model(model, "ridge_proof")

    print("RMSE (proof_level):", rmse)

2025/12/24 14:08:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/24 14:08:07 INFO mlflow.store.db.utils: Updating database tables
2025/12/24 14:08:07 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/24 14:08:07 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/24 14:08:07 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2025/12/24 14:08:07 INFO alembic.runtime.migration: Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
2025/12/24 14:08:07 INFO alembic.runtime.migration: Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
2025/12/24 14:08:07 INFO alembic.runtime.migration: Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
2025/12/24 14:08:07 INFO alembic.runtime.migration: Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
2025/12/24 14:08:07 INFO alembic.runtime.migration: Running 

ValueError: could not convert string to float: 'high'

In [ ]:
with mlflow.start_run(run_name="logistic_emotion"):

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X, y_emotion)

    preds = clf.predict(X)
    acc = accuracy_score(y_emotion, preds)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_metric("acc_emotion", acc)
    mlflow.sklearn.log_model(clf, "logistic_emotion")

    print("Accuracy (emotion_level):", acc)

In [ ]:
print("""
해석 가이드:

1) RMSE / Accuracy가 의미 있게 나오면
   → tone_vector → 파라미터 보정 가능

2) 성능이 낮으면
   → rule-based 고정 유지 (ML 불필요)

3) 혼합 전략 가능:
   rule_value + α * ml_adjustment
""")